In [1]:
import json
import os
import re
import unicodedata
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
def strip_accents_and_lowercase(s: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn"
    ).lower()

In [3]:
def write_json(data: Any, filepath: str) -> None:
    with open(filepath, "w", encoding="utf8") as fp:
        json.dump(data, fp, indent=4, ensure_ascii=False)

In [4]:
def open_json(filepath: str) -> Any:
    with open(filepath, "r", encoding="utf8") as fp:
        data = json.load(fp)
    return data

In [5]:
data_dir_path = os.path.join(".", "..", "data")
raw_dir_path = os.path.join(data_dir_path, "raw")
processed_dir_path = os.path.join(data_dir_path, "processed")
splits_dir_path = os.path.join(data_dir_path, "splits")

os.makedirs(processed_dir_path, exist_ok=True)
os.makedirs(splits_dir_path, exist_ok=True)

dataset: pd.DataFrame = pd.read_pickle(os.path.join(raw_dir_path, "dataset.pkl"))

In [6]:
dataset.shape

(33289, 8)

In [ ]:
labels_to_keep = []

In [ ]:
dataset[dataset["label_name"].isin(labels_to_keep)]["label_name"].value_counts()

# Process the dataset

In [ ]:
# Skip the following two cells if you want to keep all the labels
not_relevant = dataset[~(dataset["label_name"].isin(labels_to_keep))]

In [ ]:
not_relevant["label_name"].unique()

In [14]:
# Convert to None the labels we wish to use as the "other" class.
# We are going to model this task as a multilabel classification task, so the None will just be empty vectors.
# If it is a multiclass classification task instead of None use "other" as the label name.
dataset.loc[~(dataset["label_name"].isin(labels_to_keep)), "label_name"] = None

In [7]:
label2id = {label: idx for idx, label in enumerate(dataset[~(dataset["label_name"].isna())]["label_name"].unique())}
write_json(data=label2id, filepath=os.path.join(processed_dir_path, "label2id.json"))

In [ ]:
label2id

In [8]:
id2label = {idx: label for label, idx in label2id.items()}
write_json(data=id2label, filepath=os.path.join(processed_dir_path, "id2label.json"))

In [ ]:
id2label

In [9]:
dataset["label"] = dataset["label_name"].apply(lambda x: label2id.get(x, None))

In [ ]:
dataset.head()

## Find symbols to remove

In [47]:
all_chars = set()

for text in dataset["text"].tolist():
    text = strip_accents_and_lowercase(s=text)
    all_chars.update(text)

In [60]:
symbols_to_remove = [symbol for symbol in all_chars if not bool(re.search(r"[\w\d\s\.]", symbol))]
symbols_to_remove

['{',
 '-',
 ')',
 '’',
 ']',
 ',',
 '"',
 '«',
 '|',
 '[',
 '᾿',
 ';',
 '͵',
 '}',
 '!',
 ':',
 '(',
 '/',
 '“',
 "'",
 '΄',
 '᾽',
 '»',
 '”']

## Clean the texts

In [10]:
def remove_redundant_spaces(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

In [11]:
def clean_text(text: str, symbols_to_remove: Optional[str] = None) -> str:
    if symbols_to_remove:
        text = re.sub(symbols_to_remove, " ", text)

    text = re.sub(r"\.{2,}", ". ", text)
    text = strip_accents_and_lowercase(s=text)
    text = remove_redundant_spaces(text)
    return text

In [12]:
symbols_to_remove = r"[`~!@#$%^&*()\-+=\[\]{\}/?><,\'\":;|»«§°·¦ʼ¬£€©΄”¨•“’‘´\\…\n]"

In [13]:
dataset["processed_text"] = dataset["text"].apply(lambda x: clean_text(text=x, symbols_to_remove=symbols_to_remove))

In [ ]:
dataset.head()

In [14]:
dataset.to_pickle(os.path.join(processed_dir_path, "dataset.pkl"))

# Split the dataset

In [15]:
dataset: pd.DataFrame = pd.read_pickle(os.path.join(processed_dir_path, "dataset.pkl"))

In [ ]:
dataset.head()

In [ ]:
dataset["label"].value_counts(dropna=False)

In [17]:
y_label = dataset["label"].fillna(dataset["label"].max() + 1)

training_df, temp_df = train_test_split(dataset, test_size=0.3, random_state=0, stratify=y_label)

In [18]:
y_label = temp_df["label"].fillna(dataset["label"].max() + 1)

test_df, temp_df = train_test_split(temp_df, test_size=0.667, random_state=0, stratify=y_label)

In [19]:
# Create two validation sets, one for classification and one for contrastive learning, to avoid data leakage.

y_label = temp_df["label"].fillna(dataset["label"].max() + 1)

cl_val_df, classification_val_df = train_test_split(temp_df, test_size=0.5, random_state=0, stratify=y_label)

In [20]:
assert training_df.shape[0] + test_df.shape[0] + cl_val_df.shape[0] + classification_val_df.shape[0] == dataset.shape[0]

In [21]:
training_df.to_pickle(os.path.join(splits_dir_path, "train.pkl"))
test_df.to_pickle(os.path.join(splits_dir_path, "test.pkl"))
cl_val_df.to_pickle(os.path.join(splits_dir_path, "cl_validation.pkl"))
classification_val_df.to_pickle(os.path.join(splits_dir_path, "validation.pkl"))